# 05 · Süre tahmini deneyi: veri sözleşmesi ve medyan referansı

`ml_experiments.duration_evaluation`, TabPFN gibi bir modeli denemeden önce veri setinin güvenli olduğunu denetler
ve her zaman **medyan referansını** hesaplar. Model ancak bu referansı geçerse anlamlıdır.

> Bu notebook yalnız `data/examples/duration_experiment_synthetic.json` **sentetik** verisini kullanır.
> Gerçek Production Analysis verisi ve TabPFN deneyi henüz yapılmadı (Yunus'un veri dalı birleştirilince).

In [1]:
# Notebook'u tutorials/ veya repo kökünden açabilirsiniz; importlar repo kökünden çalışır.
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "tutorials":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import plotly.io as pio
pio.renderers.default = "notebook_connected"   # plotly.js'i her notebook'a gömmez; görüntülemek için internet gerekir
print("Repo kökü:", ROOT)

Repo kökü: /Users/nazlsu/Desktop/PickWise


In [2]:
import json
import pandas as pd
from pydantic import ValidationError

from ml_experiments.duration_evaluation import DurationDataset, evaluate_duration

raw = json.loads(Path("data/examples/duration_experiment_synthetic.json").read_text(encoding="utf-8"))
dataset = DurationDataset.model_validate(raw)
print("Kaynak:", dataset.source, "| hedef:", dataset.target_definition)
pd.DataFrame([r.model_dump() | {"bölüm": "train"} for r in dataset.train] +
             [r.model_dump() | {"bölüm": "test"} for r in dataset.test])

Kaynak: SYNTHETIC DEMO - not Production Analysis | hedef: recorded_elapsed_minutes


,record_id,job_id,part_desc,operation_type,machine_id,batch_size,started_at,finished_at,duration_minutes,bölüm
0,r1,j1,synthetic_part,turning,M1,1,2026-10-16 08:00:00+03:00,2026-10-16 08:10:00+03:00,10.0,train
1,r2,j2,synthetic_part,turning,M1,1,2026-10-16 09:00:00+03:00,2026-10-16 09:20:00+03:00,20.0,train
2,r3,j3,synthetic_part,turning,M1,1,2026-10-16 10:00:00+03:00,2026-10-16 10:25:00+03:00,25.0,test
3,r4,j4,synthetic_part,unseen,M1,1,2026-10-16 11:00:00+03:00,2026-10-16 11:05:00+03:00,5.0,test


## 1. Medyan referansı

In [3]:
report = evaluate_duration(dataset)
print("Eğitim / test örnek:", report["train_samples"], "/", report["test_samples"])
print("Bölme:", report["split_method"])
print("Medyan metrikleri:", report["median"])
print("Görülmeyen kategori:", report["fallback"])

Eğitim / test örnek: 2 / 2
Bölme: provided job-disjoint chronological holdout; all training outcomes precede test starts
Medyan metrikleri: {'mae_minutes': 10.0, 'rmse_minutes': 10.0, 'sample_count': 2}
Görülmeyen kategori: Unseen operation_type -> training global median


## 2. Sözleşme neyi reddeder?

Her deneme bir kuralı bozar; veri sessizce düzeltilmez, reddedilir.

In [4]:
import copy

def try_dataset(label, change):
    data = copy.deepcopy(raw)
    change(data)
    try:
        DurationDataset.model_validate(data)
        print(f"{label}: kabul edildi")
    except ValidationError as error:
        print(f"{label}: {error.errors()[0]['msg']}")

try_dataset("Hedef tanımı insan tarafından onaylanmamış", lambda d: d.update(target_reviewed=False))
try_dataset("Aynı iş emri hem eğitimde hem testte", lambda d: d["test"][0].update(job_id=d["train"][0]["job_id"]))
try_dataset("Süre, başlangıç-bitiş farkıyla uyuşmuyor", lambda d: d["train"][0].update(duration_minutes=99))
try_dataset("Aynı kayıt kimliği iki kez", lambda d: d["test"][0].update(record_id=d["train"][0]["record_id"]))

Hedef tanımı insan tarafından onaylanmamış: Value error, Human target-definition review is required before evaluation
Aynı iş emri hem eğitimde hem testte: Value error, Same job occurs in train and test
Süre, başlangıç-bitiş farkıyla uyuşmuyor: Value error, Duration differs from elapsed time for elapsed target; inspect span meaning
Aynı kayıt kimliği iki kez: Value error, Duplicate record IDs


## Özet

- İş emri bazlı ve zaman sıralı bölme zorunlu; aynı iş eğitim ve testte olamaz.
- `recorded_elapsed_minutes` hedefi gerçekten geçen süreyle eşleşmeli (span anlamı ayrıca doğrulanmalı).
- Model eklenecekse `evaluate_duration(dataset, predictor=..., model_metadata={"name", "version"})` aynı veride
  medyanla karşılaştırır. Planlamada varsayılan süre **kullanıcı süresi** olarak kalır.